# AI Governance and Responsible AI: Hands-on Guide with Gemini

AI Governance and Responsible AI (RAI) ensure that AI technologies are developed and deployed in a manner that is safe, ethical, transparent, and aligned with human values.

### Key Pillars of Responsible AI:
1. **Safety & Policy Alignment**: Preventing the generation of harmful, toxic, or dangerous content.
2. **Fairness & Bias Mitigation**: Minimizing stereotypical or discriminatory behaviors.
3. **Transparency & Explainability**: Understanding why and how models make decisions.
4. **Robustness & Predictability**: Ensuring the outputs are structured and reliable enough for downstream automated verification.

In this notebook, we'll demonstrate how to enforce these practices using the **Google GenAI SDK** and the `gemini-2.5-flash` model.

## 1. Environment Setup

First, install the modern `google-genai` library and initialize the Gemini client. Make sure you have added your API key to Colab's Secrets manager under the key `GOOGLE_API_KEY`.

In [ ]:
!pip --quiet install google-genai pydantic -q

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

# Initialize the client securely using Colab Secrets
try:
    api_key = userdata.get('gemini_key')
    client = genai.Client(api_key=api_key)
    print("Gemini Client initialized successfully.")
except Exception as e:
    print(f"Initialization error: {e}\nEnsure GOOGLE_API_KEY is configured in Colab Secrets.")

Gemini Client initialized successfully.


## 2. Enforcing Safety Thresholds

Gemini allows developers to adjust safety thresholds to block content that violates policies regarding harassment, hate speech, sexually explicit material, or dangerous content. This is a foundational pillar of **AI Governance**.

In [ ]:
def generate_with_safety_governance(prompt: str) -> str:
    """
    Generates content using Gemini while explicitly configuring safety settings
    to block potentially harmful material according to governance policies.
    """
    # Define strict safety boundaries
    safety_settings = [
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        ),
    ]

    config = types.GenerateContentConfig(
        safety_settings=safety_settings,
        temperature=0.2 # Lower temperature for more consistent and responsible outputs
    )

    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=prompt,
            config=config
        )
        return response.text
    except Exception as e:
        return f"Request blocked or failed due to governance safety policies: {e}"

# Test the governed execution
test_prompt = "Explain how to build a healthy culture of trust and ethical compliance in a modern engineering team."
print(generate_with_safety_governance(test_prompt))

Request blocked or failed due to governance safety policies: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


## 3. System Instructions for Alignment & Bias Mitigation

System instructions are highly effective for aligning model behavior. They force the model to adopt a neutral, objective, and unbiased persona, which is crucial for fair decision-making.

In [ ]:
def evaluate_bias_neutrality(user_scenario: str) -> str:
    """
    Instructs the LLM to act as an objective, unbiased governance auditor
    evaluating a workplace or organizational policy scenario.
    """
    system_instruction = (
        "You are an expert AI Governance Officer. Your role is to analyze scenarios "
        "for potential bias, lack of inclusivity, and ethical risks. Always maintain a neutral, "
        "constructive tone. Avoid taking personal sides; evaluate strictly against standard "
        "diversity, equity, and ethical guidelines."
    )

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.1
    )

    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=f"Evaluate this scenario: {user_scenario}",
        config=config
    )
    return response.text

scenario = "An engineering department decides to filter out resumes that don't list a prestigious computer science degree as a primary filter to save time."
print(evaluate_bias_neutrality(scenario))

### **Executive Summary**

The practice of using university prestige as a primary automated or manual filter for engineering resumes introduces significant ethical, demographic, and talent-acquisition risks. While the motivation—improving recruitment efficiency—is a valid operational objective, using institutional prestige as a crude proxy for capability creates a high likelihood of **disparate impact**, reinforces systemic socioeconomic biases, and lowers overall construct validity in the hiring process.

---

### **Key Areas of Risk Analysis**

#### **1. Diversity, Equity, and Inclusion (DEI) Risks**
*   **Socioeconomic Bias:** Enrollment at elite or highly prestigious institutions correlates strongly with legacy status, family wealth, and access to secondary education resources. Filtering by prestige implicitly favors candidates from higher socioeconomic backgrounds.
*   **Demographic Exclusion:** Historically Underrepresented Groups (HURGs)—including racial minorities, first-genera

## 4. Structured Outputs for Auditability and Consistency

For enterprise AI governance, human or automated downstream systems must be able to audit AI evaluations reliably. Getting unstructured text from an LLM is hard to parse. Using Pydantic, we can enforce a rigid JSON schema, making the AI's evaluations programmatic and structured.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# Define the structured audit schema
class RiskAssessment(BaseModel):
    has_ethical_risk: bool = Field(description="True if there are any clear ethical or safety issues found.")
    risk_level: str = Field(description="The risk classification. Must be one of: 'Low', 'Medium', 'High'.")
    detected_biases_or_violations: List[str] = Field(description="List of specific ethical principles or safety standards violated.")
    remediation_steps: List[str] = Field(description="Actionable suggestions to align this system with responsible AI practices.")

def perform_structured_governance_audit(system_description: str) -> RiskAssessment:
    """
    Uses Gemini's Structured Outputs feature to return a reliable, schema-validated
    risk assessment object representing an automated AI governance check.
    """
    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=f"Analyze the following AI use case and output the compliance schema: {system_description}",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RiskAssessment,
            temperature=0.0 # Extreme consistency
        ),
    )

    # The response.text will strictly follow our RiskAssessment model schema
    return RiskAssessment.model_validate_json(response.text)

# Example of auditing a dangerous deployment plan
ai_system_plan = (
    "We plan to deploy an emotion detection AI system using computer vision in the office "
    "to measure employee focus and automatically flag individuals who look tired or unproductive."
)

audit_result = perform_structured_governance_audit(ai_system_plan)

# Display structured results beautifully
import json
print(json.dumps(audit_result.model_dump(), indent=2))

{
  "has_ethical_risk": true,
  "risk_level": "High",
  "detected_biases_or_violations": [
    "Workplace privacy and surveillance violations",
    "Scientific invalidity of inferring productivity or internal emotional states from facial expressions",
    "Potential discrimination against neurodivergent individuals, people with disabilities, or diverse cultural backgrounds",
    "Non-compliance with AI regulations such as the EU AI Act prohibition on workplace emotion recognition"
  ],
  "remediation_steps": [
    "Immediately halt the planned deployment of computer vision emotion detection in the workplace.",
    "Transition to objective, outcome-based performance metrics rather than continuous biometric monitoring.",
    "Perform a comprehensive Data Protection Impact Assessment (DPIA) and consult with legal experts regarding labor rights.",
    "Engage in transparent communication with employee representatives and labor unions regarding workplace technology policies."
  ]
}
